# UEMR chốt, file 01 §5: train lại UniAV HT-Step với thầy Omni `best`

Theo `docs/uemr_chot/UEMR_CHOT_01_UNIAV_BEST.md` §5. Code HanLi `77fc4fa`, **không sửa**, `config.py` mặc định (kiến trúc của nó đã
kiểm khớp `model_best.pth.tar` cũ bằng `load_state_dict(strict=True)`). **Chỉ đổi vector thầy:** thư mục `datasets/omni_av_feats` (đường mặc
định của config) chứa bộ thầy mới `GraceFeel/htstep-omni-av-best` thay cho `nguyenminh04/htstep-omni-av`. Không `--set` nào khác.

- Colab Secrets: `HF_TOKEN` của `GraceFeel` (bộ thầy là dataset private). GPU: A100.
- Checkpoint ghi thẳng lên Drive (`uemr_chot_run/uniav_best/<RUN>/`). `train.py` không có resume: mất runtime thì Run all lại từ đầu.
- **Chạy thử trước:** `SMOKE = True` train 1 epoch (`opt.epochs=0`, `opt.warmup_epochs=1`), chỉ để kiểm đường chạy; checkpoint này không dùng.
  Xong đổi `SMOKE = False`, Run all.

In [ ]:
# ===== 0. Cấu hình =====
SMOKE      = True        # True: 1 epoch, ghi vào thư mục smoke riêng. Xong đổi False rồi Run all
HANLI_REF  = '77fc4fa'
RUN        = 'htstep_best'
HF_TEACHER = 'GraceFeel/htstep-omni-av-best'
TEACHER_CONFIG_SHA = 'f9abbce3876926d96d531de27e5469078d4f728c49ff5b0de2378df9335512b3'   # config.json của bộ thầy (log A2, ô 10)
BEST_ADAPTER_SHA   = '5e9899aa043191a87abdfd92f19adf89282c0b3318ab38c40c41354faa9e208a'
HTSTEP_COMMIT, ANN_SHA, TAX_SHA = 'cca1dea', 'e40d8a2b8da8024f6ea81b09fdd3bb33172b2daf7e28f3f9e581263dc77e9733', 'f19e254020848a1bdb287810586eba3d2592706fc58071cb82199ba1b767ee66'
HF_OUT     = 'GraceFeel/uniav-htstep-best'   # model repo private nhận checkpoint mới; '' = chỉ Drive
MAP_MIN    = 20.0        # plan 01 §5: mAP val (article GT) < 20 sau khi xong thì dừng, báo

In [ ]:
# ===== 1. Drive, code HanLi, GPU =====
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, re
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
T_START = time.time()
HANLI = '/content/HanLi-Retriever'

def check(name, ok, detail=''):
    print(f"CHECK {name}: {'OK' if ok else 'FAIL'}" + (f' | {detail}' if detail else ''))
    assert ok, f'CHECK {name} FAIL: {detail}'

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def wj(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + '.tmp', 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
    os.replace(path + '.tmp', path)

os.chdir('/content')
shutil.rmtree(HANLI, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/im-xiaoming/HanLi-Retriever.git', HANLI], check=True)
subprocess.run(['git', '-C', HANLI, 'checkout', '-q', HANLI_REF], check=True)
HANLI_COMMIT = subprocess.run(['git', '-C', HANLI, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()
print('HanLi:', HANLI_COMMIT)
check('HanLi đúng commit, không sửa', HANLI_COMMIT.startswith(HANLI_REF) and not subprocess.run(['git', '-C', HANLI, 'status', '--porcelain'], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pandas', 'joblib'], check=True)
import numpy as np, torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'gpu': GPU, 'hanli': HANLI_COMMIT, 'smoke': SMOKE}
print(json.dumps(ENV, ensure_ascii=False))
check('GPU A100', 'A100' in GPU, GPU or 'không có GPU')
OUT = f'/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr_chot_run/uniav_best/{RUN}' + ('_smoke' if SMOKE else '')
if SMOKE:
    shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)
DS = f'{HANLI}/datasets'
os.makedirs(DS, exist_ok=True)

In [ ]:
# ===== 2. Dữ liệu: annotation chính thức, vector text bước, đặc trưng K400 + BEATs (như lượt cũ), thầy MỚI =====
from huggingface_hub import hf_hub_download, snapshot_download
t0 = time.time()
HT = '/content/htstep_official'
if not os.path.isdir(f'{HT}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/facebookresearch/htstep', HT], check=True)
subprocess.run(['git', '-C', HT, 'checkout', '-q', HTSTEP_COMMIT], check=True)
for f, want in (('annotations.json', ANN_SHA), ('taxonomy.csv', TAX_SHA)):
    shutil.copy2(f'{HT}/data/{f}', f'{DS}/{f}')
    check(f'sha256 {f} chính thức', sha256(f'{DS}/{f}') == want)
shutil.copy2(hf_hub_download('nguyenminh04/htstep-text-feats', 'text_feats/step_emb_iv2j.npz', repo_type='dataset'), f'{DS}/step_emb_iv2j.npz')
print('step_emb_iv2j.npz sha256', sha256(f'{DS}/step_emb_iv2j.npz'))

# đặc trưng: đúng lệnh của colab_train.sh, chỉ phần k400 (KHÔNG tải thầy cũ)
if not os.path.exists(f'{DS}/k400_feats/.done'):
    r = subprocess.run([sys.executable, 'tools/fetch_htstep.py', '--what', 'k400', '--cache', '/content/hf_cache'], cwd=HANLI, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    check('tải đặc trưng K400 + BEATs', r.returncode == 0)
    open(f'{DS}/k400_feats/.done', 'w').close()
print('k400_feats:', len(glob.glob(f'{DS}/k400_feats/*.npz')), 'file')

# thầy mới, đặt ở đường mặc định của config (dataset.omni_emb_folder = ./datasets/omni_av_feats)
TEACH = f'{DS}/omni_av_feats'
shutil.rmtree(TEACH, ignore_errors=True)
snapshot_download(HF_TEACHER, repo_type='dataset', local_dir=TEACH, max_workers=16)
check('sha256 config.json của bộ thầy = log A2', sha256(f'{TEACH}/config.json') == TEACHER_CONFIG_SHA, sha256(f'{TEACH}/config.json')[:16])
TC = json.load(open(f'{TEACH}/config.json', encoding='utf-8'))
check('bộ thầy là Omni best (adapter 5e9899aa…), không phải SMOKE', TC['adapter_sha256'] == BEST_ADAPTER_SHA and TC['smoke'] is False)
bad = [k for k, h in TC['sha256'].items() if not os.path.exists(f'{TEACH}/{k}') or sha256(f'{TEACH}/{k}') != h]
n_files = {s: len(glob.glob(f'{TEACH}/{s}/*.npz')) for s in ('train', 'val')}
check('sha256 từng file thầy = config.json', not bad and sum(n_files.values()) == len(TC['sha256']), f'{n_files}, sai {bad[:5]}')
print('thầy:', n_files, '| đoạn thiếu train:', TC['counts']['train']['segments_missing'], f"({TC['counts']['train']['missing_pct']:.2f} %)", f'| {time.time() - t0:.0f} s')

In [ ]:
# ===== 3. tools/validate_htstep.py của HanLi (dừng nếu có lỗi chặn) =====
r = subprocess.run([sys.executable, 'tools/validate_htstep.py', '--workers', '8', '--report', f'{OUT}/validation_report.json'], cwd=HANLI, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-1500:])
check('validate_htstep.py: không có lỗi chặn', r.returncode == 0, f'xem {OUT}/validation_report.json')

In [ ]:
# ===== 4. Train (config mặc định 77fc4fa; checkpoint ghi thẳng lên Drive) =====
CK = f'{HANLI}/ckpt/{RUN}'
os.makedirs(f'{HANLI}/ckpt', exist_ok=True)
if os.path.lexists(CK):
    os.remove(CK) if os.path.islink(CK) else shutil.rmtree(CK)
os.symlink(OUT, CK)          # train.py ghi ckpt/<RUN>/model_best.pth.tar -> Drive
EXTRA = ['--set', 'opt.epochs=0', 'opt.warmup_epochs=1', '--patience', '0'] if SMOKE else []
CMD = [sys.executable, '-u', 'train.py', '--output', RUN, *EXTRA]
print('lệnh:', ' '.join(CMD[1:]))
LOG = f'{OUT}/train.log'
t0 = time.time()
with open(LOG, 'w', encoding='utf-8') as f:
    p = subprocess.Popen(CMD, cwd=HANLI, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        f.write(line); f.flush()
    rc = p.wait()
TRAIN_SEC = time.time() - t0
log = open(LOG, encoding='utf-8').read()
check('train.py thoát bình thường', rc == 0, f'mã {rc}, xem {LOG}')
check('loss không NaN / inf', not re.search(r'loss (nan|inf)', log, re.I))
runs = sorted(glob.glob(f'{HANLI}/experiments/runs/*.json'), key=os.path.getmtime)
RECORD = json.load(open(runs[-1])) if runs else {}
if runs:
    shutil.copy2(runs[-1], f'{OUT}/run_record.json')
print(f'train: {TRAIN_SEC / 60:.1f} phút')

In [ ]:
# ===== 5. Kết quả: checkpoint, sha256, mAP (article GT) >= 20, bảng loss / chỉ số theo epoch =====
import pandas as pd
CKPT = f'{OUT}/model_best.pth.tar'
check('có model_best.pth.tar', os.path.exists(CKPT))
ck = torch.load(CKPT, map_location='cpu', weights_only=True)
CK_SHA = sha256(CKPT)
M = ck['metrics']
print('epoch tốt nhất (mAP):', ck['epoch'], '| sha256', CK_SHA)
print(json.dumps({k: round(float(v), 2) for k, v in M.items()}, ensure_ascii=False))
hist = pd.DataFrame(RECORD.get('history', []))
if len(hist):
    cols = [c for c in ('epoch', 'seconds', 'final_loss', 'ev_loss', 'reg_loss', 'iou_loss', 'emb_loss', 'omni_point_kd', 'omni_point_cos',
                        'omni_span_kd', 'omni_span_cos', 'span_loss', 'art_loss', 'mAP', 'ArtAcc', 'OmniCos', 'OmniCosC', 'OmniR@1') if c in hist]
    print(hist[cols].round(4).to_string(index=False))
shutil.copy2(f'{HANLI}/libs/core/config.py', f'{OUT}/config.py')
SUMMARY = {'checkpoint': CKPT, 'sha256': CK_SHA, 'epoch': ck['epoch'], 'metrics': M, 'hanli': HANLI_COMMIT, 'command': ' '.join(CMD[1:]),
           'teacher': {'hf': HF_TEACHER, 'config_sha256': TEACHER_CONFIG_SHA, 'files': n_files}, 'config_py_sha256': sha256(f'{HANLI}/libs/core/config.py'),
           'train_minutes': round(TRAIN_SEC / 60, 1), 'env': ENV, 'smoke': SMOKE, 'created': time.strftime('%Y-%m-%d %H:%M:%S')}
wj(f'{OUT}/summary.json', SUMMARY)
if not SMOKE:
    check(f'mAP val (article GT) >= {MAP_MIN} (plan 01 §5)', M['mAP'] >= MAP_MIN, f"{M['mAP']:.2f}")
    if HF_OUT:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(HF_OUT, private=True, exist_ok=True)
        for f in ('model_best.pth.tar', 'config.py', 'train.log', 'run_record.json', 'summary.json'):
            if os.path.exists(f'{OUT}/{f}'):
                api.upload_file(path_or_fileobj=f'{OUT}/{f}', path_in_repo=f, repo_id=HF_OUT, commit_message=f'UniAV HT-Step, thầy Omni best (HanLi {HANLI_REF})')
        print('đã upload:', HF_OUT)
print('Gửi agent: output ô 2, 3, 5 (hoặc file notebook đã lưu output) và', f'{OUT}/summary.json')
if SMOKE:
    print('SMOKE xong: mọi CHECK OK. Đổi SMOKE = False ở ô 0 rồi Run all.')